# Graph RAG — entity-centric retrieval over a knowledge graph

Naive RAG retrieves *documents*. **Graph RAG** (Microsoft, 2024) first builds a knowledge graph of *entities* extracted from the corpus, clusters the graph into topical communities, and answers questions either at the entity level (local) or the community level (global).

This notebook walks the offline-deterministic version: an LLM extracts entities per abstract, we connect entities that co-occur in the same paper, run NetworkX's greedy modularity community detection, and let the LLM summarize each community.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import json
import networkx as nx
from shared.llm import Message, complete
from shared.loaders import load_corpus

MODEL = 'openai/gpt-4o-mini'
NS = '01-rag/09-graph-rag'
DOCS = load_corpus()
print('docs:', len(DOCS))

docs: 10


## Stage 1 — entity extraction (one call per abstract)

In [3]:
EXTRACT_SYS = (
    'Extract the technical entities mentioned in the abstract below. '
    'Return ONLY a JSON array of short strings. No prose, no markdown. '
    'Include method names, model names, datasets, and key technical concepts. '
    'Keep it to 3-6 entities.'
)
def extract_entities(abstract):
    raw = complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=EXTRACT_SYS),
        Message(role='user', content=abstract),
    ]).content
    return json.loads(raw)

per_doc = {d.arxiv_id: extract_entities(d.abstract) for d in DOCS}
for did, ents in list(per_doc.items())[:3]:
    print(did, '->', ents)

synth-001 -> ['RA-MoE', 'Mixture-of-Experts', 'expert routing', 'Llama-3']
synth-002 -> ['HA-Retrieve', 'long-context retrieval', 'cross-encoder', 'BGE']
synth-003 -> ['Polyglot-Switch', 'code-switching', 'byte-pair encoding', 'low-resource languages']


## Stage 2 — build the co-mention graph

Nodes: entities. Edges: same paper. Weighted by number of co-mentions.

In [4]:
G = nx.Graph()
for did, ents in per_doc.items():
    for e in ents:
        G.add_node(e)
    for i, a in enumerate(ents):
        for b in ents[i+1:]:
            w = G[a][b]['weight'] + 1 if G.has_edge(a, b) else 1
            G.add_edge(a, b, weight=w)
print('nodes:', G.number_of_nodes(), '  edges:', G.number_of_edges())

nodes: 36   edges: 60


## Stage 3 — community detection

Greedy modularity is fast and parameter-free; production stacks use Leiden.

In [5]:
communities = list(nx.algorithms.community.greedy_modularity_communities(G))
ent_to_doc = {e: [d for d, ents in per_doc.items() if e in ents] for e in G.nodes}
comm_docs = []
for c in communities:
    docs = sorted({d for e in c for d in ent_to_doc.get(e, [])})
    comm_docs.append({'entities': sorted(c), 'docs': docs})
for i, cd in enumerate(comm_docs):
    print(f'community {i}: {len(cd["entities"])} entities, {len(cd["docs"])} docs')

community 0: 10 entities, 3 docs
community 1: 7 entities, 2 docs
community 2: 7 entities, 2 docs
community 3: 4 entities, 1 docs
community 4: 4 entities, 1 docs
community 5: 4 entities, 1 docs


## Stage 4 — summarize each community (LLM)

In [6]:
SUMMARY_SYS = (
    'Summarize the following cluster of related technical entities and the papers that '
    'mention them. One paragraph, under 80 words, neutral tone.'
)
def summarize(community):
    user = (
        f"Entities: {json.dumps(community['entities'])}\n"
        f"Papers (by id): {json.dumps(community['docs'])}\n\n"
        'Summarize this community in one paragraph.'
    )
    return complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=SUMMARY_SYS),
        Message(role='user', content=user),
    ]).content

for cd in comm_docs:
    print('---', cd['docs'], '---')
    print(summarize(cd))
    print()

--- ['synth-003', 'synth-005', 'synth-009'] ---
This community centres on FLORES-200, Polyglot-Switch, Tokenizer-Equity Index, byte-pair encoding, code-switching, low-resource languages. The cluster is grounded in papers synth-003, synth-005, synth-009, which share these technical concepts. Together they outline a coherent line of work in the cs.CL literature.

--- ['synth-001', 'synth-004'] ---
This community centres on AdaSpec, Llama-3, Mixture-of-Experts, RA-MoE, draft length, expert routing. The cluster is grounded in papers synth-001, synth-004, which share these technical concepts. Together they outline a coherent line of work in the cs.CL literature.

--- ['synth-002', 'synth-007'] ---
This community centres on BGE, HA-Retrieve, Listwise-Aligned BGE, bi-encoder, cross-encoder, long-context retrieval. The cluster is grounded in papers synth-002, synth-007, which share these technical concepts. Together they outline a coherent line of work in the cs.CL literature.

--- ['synth-006

## What you can extend

* Replace `greedy_modularity_communities` with Leiden (`python-igraph`) for better cluster quality.
* Add an entity-relation extraction prompt (`subject, verb, object`) and build a directed graph.
* Implement Microsoft's **global vs local** retrieval: route concept-level queries to community summaries, entity-level queries to the entity neighbourhood.